In [3]:
import pandas as pd
import numpy as np
import wrds as wd

In [4]:
df_unique = pd.read_csv("df_unique_sorted.csv")
df_sst = pd.read_csv("SST.v5.csv")

C:\Users\conno\AppData\Local\Temp\ipykernel_17444\3082282194.py:2: DtypeWarning: Columns (96,101,103) have mixed types. Specify dtype option on import or set low_memory=False.
  df_sst = pd.read_csv("SST.v5.csv")


In [5]:
df_unique.columns

Index(['CCN', 'Hospital_Name', 'State', 'City', 'Zip', 'County', 'Address',
       'Year', 'FY_Begin', 'FY_End',
       ...
       'full_address', 'location', 'latitude', 'longitude', 'nearest_building',
       'nearest_building_road_dist_miles', 'nearest_building_1',
       'nearest_building_1_road_dist_miles', 'nearest_building_2',
       'nearest_building_2_road_dist_miles'],
      dtype='object', length=119)

In [6]:
df_sst['CCN'].nunique()

5094

# Merge nearest_building_road_dist_miles into SST_v4\n\nSource: `updated_df_road_dist.csv` (5-state region: AR, LA, NM, OK, TX).\nFor the 23 CCNs with differing distance values across years, keep the most recent year's value.

In [7]:
road = pd.read_csv("../updated_df_road_dist.csv", dtype={'CCN': str, 'Year': int})

# Keep most recent year per CCN, then pull only the distance column
road_deduped = (
    road.sort_values('Year', ascending=False)
    .drop_duplicates(subset='CCN', keep='first')
    [['CCN', 'nearest_building_road_dist_miles']]
)

# Filter SST to 5-state region and left-join distance
states_5 = ['AR', 'NM', 'LA', 'OK', 'TX']
sst_5state = df_sst[df_sst['State'].isin(states_5)].copy()
sst_5state['CCN'] = sst_5state['CCN'].astype(str)

# Drop stale road-distance column if already present in upstream data
if 'nearest_building_road_dist_miles' in sst_5state.columns:
    sst_5state = sst_5state.drop(columns=['nearest_building_road_dist_miles'])

sst_v5 = sst_5state.merge(road_deduped, on='CCN', how='left')

print(f"Rows: {len(sst_v5):,}  |  Unique CCNs: {sst_v5['CCN'].nunique()}")
print(f"Distance populated: {sst_v5['nearest_building_road_dist_miles'].notna().sum():,} rows")
print(f"Distance missing:   {sst_v5['nearest_building_road_dist_miles'].isna().sum():,} rows")

# Spot-check REH converters
reh = sst_v5[sst_v5['Is_REH_Converter'] == 1][['CCN', 'Hospital_Name', 'State', 'Year', 'nearest_building_road_dist_miles']].drop_duplicates('CCN')
print(f"
REH converters with distance: {reh['nearest_building_road_dist_miles'].notna().sum()} / {len(reh)}")
reh

Rows: 3,240  |  Unique CCNs: 870
Distance populated: 2,280 rows
Distance missing:   960 rows

REH converters with distance: 24 / 24


,CCN,Hospital_Name,State,Year,nearest_building_road_dist_miles
77,40047,FIVE RIVERS MEDICAL CENTER,AR,2020,17.067575
125,40085,HELENA REGIONAL MEDICAL CENTER,AR,2020,45.193570
177,40779,FIVE RIVERS MEDICAL CENTER,AR,2024,17.067575
190,41304,EUREKA SPRINGS HOSPITAL,AR,2020,14.031245
230,41314,DEWITT HOSPITAL & NURSING HOME INC,AR,2020,27.783619
233,41316,SMC MEDICAL CENTER,AR,2020,20.183503
540,320067,GUADALUPE COUNTY HOSPITAL,NM,2020,66.988102
559,320779,GUADALUPE COUNTY HOSPITAL,NM,2024,66.988102
695,370029,ALLIANCE HEALTH CLINTON,OK,2020,17.203469
698,370030,BLACKWELL REGIONAL HOSPITAL,OK,2020,19.938372


In [8]:
from build_sst import SST_V6_COLUMNS, validate_schema

# Reset the row index as Unnamed: 0 (sequential integer for this 5-state slice)
sst_v5 = sst_v5.reset_index(drop=True)
if 'Unnamed: 0' in sst_v5.columns:
    sst_v5 = sst_v5.drop(columns=['Unnamed: 0'])
sst_v5.insert(0, 'Unnamed: 0', sst_v5.index)

# Enforce exact SST_v6 column schema
missing_cols = [c for c in SST_V6_COLUMNS if c not in sst_v5.columns]
extra_cols   = [c for c in sst_v5.columns if c not in SST_V6_COLUMNS]
if missing_cols:
    print(f"WARNING: Adding {len(missing_cols)} missing columns as NaN: {missing_cols}")
    for col in missing_cols:
        import numpy as np
        sst_v5[col] = np.nan
if extra_cols:
    print(f"WARNING: Dropping {len(extra_cols)} extra columns not in schema: {extra_cols}")
    sst_v5 = sst_v5.drop(columns=extra_cols)

sst_v5 = sst_v5[SST_V6_COLUMNS]

# Validate before saving
errors = validate_schema(sst_v5)
if errors:
    print("SCHEMA VALIDATION FAILED — fix these before saving:")
    for e in errors:
        print(f"  x {e}")
else:
    sst_v5.to_csv("SST_v6.csv", index=False)
    print(f"Saved SST_v6.csv  |  {len(sst_v5):,} rows  |  {len(sst_v5.columns)} columns")
    print(f"States: {sorted(sst_v5['State'].unique())}")
    print(f"Years:  {sorted(sst_v5['Year'].unique())}")

Saved SST_v6.csv


In [ ]:
# Round-trip schema check on the saved file
import pandas as pd
from build_sst import SST_V6_COLUMNS, validate_schema

saved = pd.read_csv("SST_v6.csv", nrows=0)
errors = validate_schema(saved, label="SST_v6.csv")
if errors:
    for e in errors: print(f"  x {e}")
else:
    print(f"SST_v6.csv schema OK — {len(SST_V6_COLUMNS)} columns in correct order")


In [ ]:
"""
generate_baselines_v5.py
Generates CAH_REH_Baselines.json from SST_v5.csv
SST_v5 is pre-filtered to the 5-state region and includes nearest_building_road_dist_miles.
Output is uploaded to GitHub raw URL for the calculator app.
"""
import pandas as pd
import numpy as np
import json
import warnings
from datetime import date

warnings.filterwarnings('ignore')

# ── Constants ─────────────────────────────────────────────────────────────────

CAH_CODE_MIN, CAH_CODE_MAX = 1300, 1399

AFP_MONTHLY = {2023: 272_866, 2024: 276_234, 2025: 285_625.90, 2026: 295_051.54}
AFP_ANNUAL  = {yr: mo * 12 for yr, mo in AFP_MONTHLY.items()}

MARGINAL_IP_COST_PCT = 0.40
OPPS_PREMIUM_PCT     = 0.05
B340B_PENALTY_PCT    = 0.02

COERCE_COLS = [
    'Total_Hospital_Expenses', 'NASHP_Net_Patient_Revenue', 'NASHP_Net_Income',
    'NASHP_Operating_Expenses', 'NASHP_Hospital_Op_Costs',
    'CMS_Inpatient_Revenue', 'CMS_Outpatient_Revenue', 'CMS_Net_Patient_Revenue',
    'CMS_Total_Costs', 'CMS_Net_Income', 'CMS_Total_Salaries',
    'CMS_Total_Assets', 'CMS_Total_Liabilities', 'CMS_Current_Assets',
    'CMS_Current_Liabilities', 'CMS_Cash',
    'Num_Beds', 'FTE_Employees', 'Total_Discharges', 'Total_Patient_Days',
    'Inpatient_Occupancy', 'Operating_Margin_Pct', 'Net_Profit_Margin_Pct',
    'Days_Cash_on_Hand', 'Current_Ratio', 'Debt_Ratio',
    'Medicare_Payer_Mix_Pct', 'Medicare_Pct_Days',
    'Medicaid_Payer_Mix_Pct', 'Commercial_Payer_Mix_Pct',
    'COVID_PHE_Funding', 'Direct_Labor_Cost', 'Contract_Labor_Cost',
    'Is_340B_Enrolled', 'Fund_Balance',
    'nearest_building_road_dist_miles',
]

BASELINE_COLS = [
    'NASHP_Net_Patient_Revenue', 'Total_Hospital_Expenses', 'NASHP_Net_Income',
    'OP_Share', 'Medicare_Payer_Mix_Pct', 'Is_340B_Enrolled',
    'Days_Cash_on_Hand', 'Current_Ratio', 'Debt_Ratio', 'Operating_Margin_Pct',
    'Num_Beds', 'FTE_Employees', 'Total_Discharges', 'Total_Patient_Days',
    'Inpatient_Occupancy', 'NonCOVID_Other_Income',
    'Direct_Labor_Cost', 'Contract_Labor_Cost', 'Fund_Balance',
]

# ── Load ───────────────────────────────────────────────────────────────────────
# SST_v5 is already filtered to AR, LA, NM, OK, TX

print("Loading SST_v6.csv ...")
df = pd.read_csv('SST_v6.csv', low_memory=False, encoding='utf-8')
df['CCN'] = df['CCN'].astype(str).str.strip().str.zfill(6)
print(f"  {len(df)} rows in 5-state region")

for col in COERCE_COLS:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

# ── Hospital Type ─────────────────────────────────────────────────────────────

df['Pre_REH_CCN'] = df['Pre_REH_CCN'].apply(
    lambda x: str(x).split('.')[0].zfill(6)
    if pd.notna(x) and str(x).lower() not in ('nan', '', '.') else np.nan
)
df['Unified_CCN'] = df['Pre_REH_CCN'].fillna(df['CCN'])
facility_code = pd.to_numeric(df['Unified_CCN'].str[2:], errors='coerce')
df['Is_True_CAH']    = facility_code.between(CAH_CODE_MIN, CAH_CODE_MAX)
df['Is_PPS_Converter'] = df['Pre_REH_CCN'].notna()

df_scope = df[df['Is_True_CAH'] | df['Is_PPS_Converter']].copy()
print(f"  {len(df_scope)} rows for CAHs/converters")

# ── Date Parsing ──────────────────────────────────────────────────────────────

df_scope['FY_End_dt'] = pd.to_datetime(
    df_scope['FY_End_NASHP'].fillna(df_scope['FY_End_CMS']), errors='coerce'
)
df_scope['FY_Begin_dt'] = pd.to_datetime(
    df_scope['FY_Begin_NASHP'].fillna(df_scope['FY_Begin_CMS']), errors='coerce'
)
df_scope['REH_Conv_dt'] = pd.to_datetime(df_scope['REH_Conversion_Date'], errors='coerce')

never_conv = df_scope['REH_Conv_dt'].isna()
df_scope.loc[never_conv, 'Row_Type'] = 'CAH_Year'

converted = ~never_conv
df_scope.loc[converted & (df_scope['FY_End_dt'] <= df_scope['REH_Conv_dt']), 'Row_Type'] = 'CAH_Year'
df_scope.loc[converted & (df_scope['FY_Begin_dt'] >= df_scope['REH_Conv_dt']), 'Row_Type'] = 'REH_Year'
df_scope.loc[converted & df_scope['Row_Type'].isna(), 'Row_Type'] = 'Transition'

# ── OP Share ──────────────────────────────────────────────────────────────────

total_gross = df_scope['CMS_Outpatient_Revenue'].fillna(0) + df_scope['CMS_Inpatient_Revenue'].fillna(0)
has_cms = (
    df_scope['CMS_Outpatient_Revenue'].notna() &
    df_scope['CMS_Inpatient_Revenue'].notna() &
    (total_gross > 0)
)
df_scope['OP_Share'] = np.where(
    has_cms,
    df_scope['CMS_Outpatient_Revenue'] / total_gross,
    df_scope['Outpatient_Rev_Pct'].fillna(np.nan) / 100
)
df_scope['OP_Share'] = df_scope['OP_Share'].clip(0, 1)

# ── Non-COVID Other Income ────────────────────────────────────────────────────

df_scope['Implied_Other_Income'] = (
    df_scope['NASHP_Net_Income'] -
    (df_scope['NASHP_Net_Patient_Revenue'] - df_scope['Total_Hospital_Expenses'])
)
df_scope['NonCOVID_Other_Income'] = (
    df_scope['Implied_Other_Income'] - df_scope['COVID_PHE_Funding'].fillna(0)
)

# ── CAH Year Rows ─────────────────────────────────────────────────────────────

df_cah_years = df_scope[df_scope['Row_Type'] == 'CAH_Year'].copy()
print(f"  {len(df_cah_years)} CAH-year rows")

# ── Solvency Classification ───────────────────────────────────────────────────

def classify_solvency(row):
    score = 0
    flags = []
    dcoh = row.get('Days_Cash_on_Hand', np.nan)
    cr   = row.get('Current_Ratio',     np.nan)
    dr   = row.get('Debt_Ratio',        np.nan)
    om   = row.get('Operating_Margin_Pct', np.nan)

    if pd.notna(dcoh):
        if dcoh < 30:   score += 2; flags.append('DCOH<30')
        elif dcoh < 90: score += 1; flags.append('DCOH<90')
    if pd.notna(cr):
        if cr < 1.0:    score += 2; flags.append('CR<1.0')
        elif cr < 2.0:  score += 1; flags.append('CR<2.0')
    if pd.notna(dr):
        if dr > 0.7:    score += 2; flags.append('DR>0.7')
        elif dr > 0.5:  score += 1; flags.append('DR>0.5')
    if pd.notna(om):
        if om < -0.10:  score += 2; flags.append('OM<-10%')
        elif om < 0:    score += 1; flags.append('OM<0%')

    if score >= 4:   status = 'Distressed'
    elif score >= 2: status = 'Marginal'
    else:            status = 'Stable'

    return score, status, ', '.join(flags) if flags else 'None'

# ── Build Baseline (Average) ──────────────────────────────────────────────────

print("Building average baseline ...")
baseline_avg = df_cah_years.groupby(['Unified_CCN', 'State'])[BASELINE_COLS].mean().reset_index()

# Get most-common hospital name, city, county
name_map = df_cah_years.groupby('Unified_CCN')['Hospital_Name'].agg(
    lambda x: x.mode().iloc[0] if len(x.mode()) > 0 else x.iloc[0]
).reset_index()
city_map = df_cah_years.groupby('Unified_CCN')['City'].agg(
    lambda x: x.mode().iloc[0] if pd.notna(x).any() else ''
).reset_index()
county_map = df_cah_years.groupby('Unified_CCN')['County'].agg(
    lambda x: x.mode().iloc[0] if pd.notna(x).any() else ''
).reset_index()

baseline_avg = baseline_avg.merge(name_map,   on='Unified_CCN', how='left')
baseline_avg = baseline_avg.merge(city_map,   on='Unified_CCN', how='left')
baseline_avg = baseline_avg.merge(county_map, on='Unified_CCN', how='left')

# Year counts
yr_counts = df_cah_years.groupby('Unified_CCN')['Year'].agg(['count','min','max']).reset_index()
yr_counts.columns = ['Unified_CCN', 'N_CAH_Years', 'First_CAH_Year', 'Last_CAH_Year']
baseline_avg = baseline_avg.merge(yr_counts, on='Unified_CCN', how='left')

# Boolean flags
type_map = df_cah_years.groupby('Unified_CCN')[['Is_True_CAH', 'Is_PPS_Converter']].first().reset_index()
baseline_avg = baseline_avg.merge(type_map, on='Unified_CCN', how='left')
baseline_avg['Is_340B_Enrolled'] = (baseline_avg['Is_340B_Enrolled'] > 0).astype(int)

# Solvency
solv_results = baseline_avg[['Days_Cash_on_Hand', 'Current_Ratio', 'Debt_Ratio', 'Operating_Margin_Pct']].apply(
    lambda row: classify_solvency(row), axis=1, result_type='expand'
)
solv_results.columns = ['Solvency_Score', 'Solvency_Status', 'Solvency_Flags']
baseline_avg = pd.concat([baseline_avg, solv_results], axis=1)

# Derived fields
baseline_avg['Historical_Op_Income'] = (
    baseline_avg['NASHP_Net_Patient_Revenue'] - baseline_avg['Total_Hospital_Expenses']
)
baseline_avg['Historical_Op_Margin'] = np.where(
    baseline_avg['NASHP_Net_Patient_Revenue'] > 0,
    baseline_avg['Historical_Op_Income'] / baseline_avg['NASHP_Net_Patient_Revenue'],
    np.nan
)
baseline_avg['IP_Revenue'] = (
    baseline_avg['NASHP_Net_Patient_Revenue'] * (1 - baseline_avg['OP_Share'].fillna(0.5))
)
baseline_avg['OP_Revenue'] = (
    baseline_avg['NASHP_Net_Patient_Revenue'] * baseline_avg['OP_Share'].fillna(0.5)
)
baseline_avg['Swing_Bed_Risk'] = (1 - baseline_avg['OP_Share'].fillna(0.5)) > 0.25

# Converter metadata
conv_meta = (
    df_scope[df_scope['Is_PPS_Converter']]
    .groupby('Unified_CCN')[['REH_Conv_dt', 'Pre_REH_CCN']].first().reset_index()
)
conv_meta.columns = ['Unified_CCN', 'REH_Conversion_Date', 'Pre_REH_CCN_raw']
baseline_avg = baseline_avg.merge(conv_meta, on='Unified_CCN', how='left')
baseline_avg['REH_Conversion_Date'] = baseline_avg['REH_Conversion_Date'].astype(str).replace('NaT', None)

# Road distance — static per hospital, take most recent non-null value
dist_map = (
    df_cah_years.sort_values('FY_End_dt')
    .groupby('Unified_CCN')['nearest_building_road_dist_miles']
    .last()
    .reset_index()
)
baseline_avg = baseline_avg.merge(dist_map, on='Unified_CCN', how='left')

print(f"  {len(baseline_avg)} hospitals in average baseline")

# ── Build Baseline (Most Recent Year) ─────────────────────────────────────────

print("Building most-recent-year baseline ...")
df_cah_sorted = df_cah_years.sort_values('FY_End_dt')
baseline_recent = df_cah_sorted.groupby('Unified_CCN').last().reset_index()
baseline_recent['Baseline_Year'] = baseline_recent['Year']

# Keep only needed cols
keep_cols = ['Unified_CCN', 'Baseline_Year'] + BASELINE_COLS
baseline_recent = baseline_recent[[c for c in keep_cols if c in baseline_recent.columns]].copy()

# Add metadata from avg
meta_cols = ['Unified_CCN', 'State', 'Hospital_Name', 'City', 'County',
             'N_CAH_Years', 'First_CAH_Year', 'Last_CAH_Year',
             'Is_True_CAH', 'Is_PPS_Converter', 'REH_Conversion_Date',
             'nearest_building_road_dist_miles']
baseline_recent = baseline_recent.merge(
    baseline_avg[[c for c in meta_cols if c in baseline_avg.columns]],
    on='Unified_CCN', how='left'
)
baseline_recent['Is_340B_Enrolled'] = (baseline_recent['Is_340B_Enrolled'] > 0).astype(int)

# Solvency
solv_r = baseline_recent[['Days_Cash_on_Hand', 'Current_Ratio', 'Debt_Ratio', 'Operating_Margin_Pct']].apply(
    lambda row: classify_solvency(row), axis=1, result_type='expand'
)
solv_r.columns = ['Solvency_Score', 'Solvency_Status', 'Solvency_Flags']
baseline_recent = pd.concat([baseline_recent, solv_r], axis=1)

# Derived fields
baseline_recent['Historical_Op_Income'] = (
    baseline_recent['NASHP_Net_Patient_Revenue'] - baseline_recent['Total_Hospital_Expenses']
)
baseline_recent['Historical_Op_Margin'] = np.where(
    baseline_recent['NASHP_Net_Patient_Revenue'] > 0,
    baseline_recent['Historical_Op_Income'] / baseline_recent['NASHP_Net_Patient_Revenue'],
    np.nan
)
baseline_recent['IP_Revenue'] = (
    baseline_recent['NASHP_Net_Patient_Revenue'] * (1 - baseline_recent['OP_Share'].fillna(0.5))
)
baseline_recent['OP_Revenue'] = (
    baseline_recent['NASHP_Net_Patient_Revenue'] * baseline_recent['OP_Share'].fillna(0.5)
)
baseline_recent['Swing_Bed_Risk'] = (1 - baseline_recent['OP_Share'].fillna(0.5)) > 0.25

print(f"  {len(baseline_recent)} hospitals in most-recent baseline")

# ── Serialize ─────────────────────────────────────────────────────────────────

def clean_for_json(df):
    records = []
    for _, row in df.iterrows():
        rec = {}
        for col, val in row.items():
            if pd.isna(val) if not isinstance(val, (bool, str)) else False:
                rec[col] = None
            elif isinstance(val, (np.integer,)):
                rec[col] = int(val)
            elif isinstance(val, (np.floating,)):
                rec[col] = None if np.isnan(val) else float(round(val, 6))
            elif isinstance(val, (np.bool_,)):
                rec[col] = bool(val)
            elif isinstance(val, pd.Timestamp):
                rec[col] = val.isoformat()
            else:
                rec[col] = val
        records.append(rec)
    return records

output = {
    "metadata": {
        "generated_from": "SST_v5.csv",
        "processing_date": date.today().isoformat(),
        "states": ["AR", "LA", "NM", "OK", "TX"],
        "total_hospitals_avg":    int(len(baseline_avg)),
        "true_cahs_avg":          int(baseline_avg['Is_True_CAH'].sum()),
        "total_hospitals_recent": int(len(baseline_recent)),
        "true_cahs_recent":       int(baseline_recent['Is_True_CAH'].sum()),
    },
    "baseline_average":     clean_for_json(baseline_avg),
    "baseline_most_recent": clean_for_json(baseline_recent),
}

out_path = 'CAH_REH_Baselines.json'
print(f"Writing {out_path} ...")
with open(out_path, 'w', encoding='utf-8') as f:
    json.dump(output, f, separators=(',', ':'), default=str)

size_kb = len(json.dumps(output).encode('utf-8')) / 1024
print(f"Done. {size_kb:.0f} KB, {len(output['baseline_average'])} avg records, "
      f"{len(output['baseline_most_recent'])} recent records")
